This is a baseline of a private comp made to the students

competition link: https://www.kaggle.com/t/fb09b38357f2432ba9e4e9ac9f340531


you are given the EMNIST dataset, which contains the 10 digits and english alphabets( both capital and small) the class is case sensitive to letters so you have 62 total classes

imagine you are trying to train a model to predict 4 character sequences from images, if you train on each unique sequence as class you would have 62^4 possible classes , not smart idea to train on sequences right?

so you are given in this comp with emnist only and should try to make use of it predict these 4 charecter combination images

In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torchvision import transforms


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_PATH = "/kaggle/input/datasets/tahaalshatiri/cnn-on-all/emnist_fast_cnn.pth"
TEST_IMAGES_DIR = "/kaggle/input/competitions/emnist-4-digit-classifier/combined_emnist_4char/images"

THRESHOLD = 20
EXPECTED_CHARS = 4


class FastEMNISTCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

classes = checkpoint["classes"]
num_classes = len(classes)

model = FastEMNISTCNN(num_classes)
model.load_state_dict(checkpoint["model"])
model = model.to(DEVICE)
model.eval()


char_transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])


def crop_to_ink(arr):
    mask = arr > THRESHOLD

    ys, xs = np.where(mask)

    if len(xs) == 0:
        return arr

    return arr[ys.min():ys.max() + 1, xs.min():xs.max() + 1]


def split_by_zero_columns(image_path):
    img = Image.open(image_path).convert("L")
    arr = np.array(img)

    binary = arr > THRESHOLD

    col_has_ink = binary.any(axis=0)

    segments = []
    inside = False
    start = None

    for x, has_ink in enumerate(col_has_ink):
        if has_ink and not inside:
            start = x
            inside = True

        elif not has_ink and inside:
            end = x
            if end - start > 1:
                segments.append((start, end))
            inside = False

    if inside:
        segments.append((start, len(col_has_ink)))

    segments = [(a, b) for a, b in segments if b - a >= 2]

    if len(segments) != EXPECTED_CHARS:
        # fallback: split ink bounding box into 4 equal chunks
        ink = crop_to_ink(arr)
        h, w = ink.shape
        chunk_w = w // EXPECTED_CHARS

        crops = []
        for i in range(EXPECTED_CHARS):
            x1 = i * chunk_w
            x2 = (i + 1) * chunk_w if i < EXPECTED_CHARS - 1 else w
            crop = crop_to_ink(ink[:, x1:x2])
            crops.append(Image.fromarray(crop))
        return crops

    crops = []

    for x1, x2 in segments:
        crop = arr[:, x1:x2]
        crop = crop_to_ink(crop)
        crops.append(Image.fromarray(crop))

    return crops


@torch.no_grad()
def predict_char(img):
    x = char_transform(img).unsqueeze(0).to(DEVICE)

    logits = model(x)
    pred_idx = logits.argmax(1).item()

    return classes[pred_idx]


def predict_image(image_path):
    crops = split_by_zero_columns(image_path)

    pred = ""
    for crop in crops:
        pred += predict_char(crop)

    return pred


rows = []

image_files = sorted([
    f for f in os.listdir(TEST_IMAGES_DIR)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
])

for i, filename in enumerate(image_files):
    image_path = os.path.join(TEST_IMAGES_DIR, filename)

    try:
        prediction = predict_image(image_path)

        if len(prediction) != 4:
            prediction = "AAAA"

    except Exception:
        prediction = "AAAA"

    rows.append([filename, prediction])

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(image_files)}")


submission = pd.DataFrame(
    rows,
    columns=["image_id", "prediction"]
)

submission.to_csv("submission.csv", index=False)

submission.head()

Processed 100/1000
Processed 200/1000
Processed 300/1000
Processed 400/1000
Processed 500/1000
Processed 600/1000
Processed 700/1000
Processed 800/1000
Processed 900/1000
Processed 1000/1000


,image_id,prediction
0,img_0000.png,2aav
1,img_0001.png,aydT
2,img_0002.png,aa7l
3,img_0003.png,vada
4,img_0004.png,EA6u
